# Week 1 — LangGraph Basics

**Phase 05 · Agents & Knowledge Graphs**

This notebook walks through the fundamentals of LangGraph — Anthropic/LangChain's framework
for building stateful, multi-step AI agents as directed graphs.

## Learning objectives
1. Understand `StateGraph`, nodes, edges, and conditional routing
2. Build a simple pension Q&A agent with two tools
3. Trace state as it flows between nodes
4. Bind LangChain tools to a LangGraph agent
5. Stream intermediate steps to the user
6. Interrupt the graph for human approval
7. Persist conversation history with `MemorySaver`


## 1. What is LangGraph?

LangGraph models an AI agent as a **directed graph**:

```
START → node_A → node_B → END
              ↘ node_C ↗
```

Key concepts:

| Concept | Description |
|---------|-------------|
| **State** | A typed dict shared across all nodes. Nodes read from it and return partial updates. |
| **Node** | A Python function `(state) → partial_state_update`. |
| **Edge** | A directed connection between nodes. |
| **Conditional edge** | A function that returns a string key; LangGraph selects the next node from a mapping. |
| **Checkpointer** | Persists state between invocations for multi-turn conversations. |

Unlike simple LLM chains, LangGraph supports **cycles** (e.g., retry loops), **parallel branches**,
and **human-in-the-loop** interrupts.


In [ ]:
# Install dependencies (skip if already installed)
# %pip install langgraph langchain-core langchain-ollama chromadb duckdb neo4j -q

In [ ]:
from __future__ import annotations

from typing import Annotated, TypedDict

from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_ollama import ChatOllama
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages

print("Imports OK")

## 2. Building a simple pension Q&A agent

We'll build a minimal agent with two nodes:
- `answer_question` — calls the LLM with a pension system prompt
- `cite_regulation` — appends a regulation citation if the answer mentions IORP

### 2a. Define the State


In [ ]:
class PensionQAState(TypedDict):
    """Minimal state for the pension Q&A agent."""
    # add_messages is a reducer: it appends new messages rather than overwriting
    messages: Annotated[list[BaseMessage], add_messages]
    needs_citation: bool   # flag set by answer_question
    citations: list[str]   # list of regulation references appended later

print("State defined:", PensionQAState.__annotations__)

### 2b. Define the LLM and tools

In [ ]:
llm = ChatOllama(model="llama3.2", temperature=0.1)

PENSION_SYSTEM = """You are an expert on Dutch and European pension regulation.
Answer questions concisely and accurately. When citing a regulation, always include
the article number (e.g. IORP III Art. 19)."""

@tool
def lookup_coverage_ratio_rule(fund_jurisdiction: str) -> str:
    """Return the coverage ratio requirement for a jurisdiction.
    
    Args:
        fund_jurisdiction: 'NL' for Netherlands, 'EU' for pan-European.
    """
    rules = {
        "NL": "FTK requires a policy coverage ratio >= 110% for unconditional indexation.",
        "EU": "IORP III Art. 19 requires coverage ratio >= 100% at all times.",
    }
    return rules.get(fund_jurisdiction.upper(), f"No data for jurisdiction '{fund_jurisdiction}'.")


@tool
def get_orsa_deadline(regulation: str) -> str:
    """Return the ORSA submission deadline for a given regulation.
    
    Args:
        regulation: 'IORP II' or 'IORP III'.
    """
    deadlines = {
        "IORP II": "Annual, within 6 months of the financial year-end.",
        "IORP III": "Annual, and ad-hoc when a significant change in risk profile occurs.",
    }
    return deadlines.get(regulation, "Unknown regulation.")

print("Tools defined:", [lookup_coverage_ratio_rule.name, get_orsa_deadline.name])

## 3. State management — how state flows between nodes

Each node returns a **partial** state update — only the keys it wants to change.
LangGraph merges updates using reducers:
- Default: overwrite the key
- `Annotated[list, add_messages]`: append to the list (used for message history)

```
State before node_A:   {messages: [H1], needs_citation: False, citations: []}
node_A returns:        {messages: [A1], needs_citation: True}
State after node_A:    {messages: [H1, A1], needs_citation: True, citations: []}
                                           ↑ merged, not replaced
```


In [ ]:
def answer_question(state: PensionQAState) -> dict:
    """Node: call the LLM and set needs_citation flag."""
    messages = [SystemMessage(content=PENSION_SYSTEM)] + state["messages"]
    response = llm.invoke(messages)
    # Check if the response mentions a regulation
    needs_citation = any(
        kw in response.content
        for kw in ["IORP", "FTK", "Article", "Art.", "directive"]
    )
    print(f"[answer_question] needs_citation={needs_citation}")
    return {"messages": [response], "needs_citation": needs_citation}


def cite_regulation(state: PensionQAState) -> dict:
    """Node: append formal citation strings to state.citations."""
    # In a real system this would parse the answer and look up citation metadata
    citations = ["IORP III (EU) 2025/XXX", "FTK, Article 131, Besluit FTK"]
    citation_note = AIMessage(
        content=f"**Regulatory references:** {', '.join(citations)}"
    )
    return {"messages": [citation_note], "citations": citations}


def skip_citation(state: PensionQAState) -> dict:
    """Node: no-op when citation is not needed."""
    return {}


def decide_citation(state: PensionQAState) -> str:
    """Conditional edge: route to cite_regulation or skip_citation."""
    return "cite_regulation" if state["needs_citation"] else "skip_citation"


print("Nodes defined.")

## 4. Tool binding — how LangChain tools work with LangGraph

Tools defined with `@tool` are callable Python functions with a `.name`, `.description`,
and `.args_schema` attached. LangGraph does **not** invoke tools automatically — the LLM must
emit a `tool_call`, and a node must execute it.

A typical pattern is to bind tools to the LLM so it knows what's available:

In [ ]:
# Bind tools to the LLM (the LLM will now know it can call these tools)
tools = [lookup_coverage_ratio_rule, get_orsa_deadline]
llm_with_tools = llm.bind_tools(tools)

# Inspect a tool's schema
import json
schema = lookup_coverage_ratio_rule.args_schema.schema()
print("Tool schema:")
print(json.dumps(schema, indent=2))

## 5. Assembling and running the graph

In [ ]:
# Build the graph
builder = StateGraph(PensionQAState)

builder.add_node("answer_question", answer_question)
builder.add_node("cite_regulation", cite_regulation)
builder.add_node("skip_citation",   skip_citation)

builder.add_edge(START, "answer_question")
builder.add_conditional_edges(
    "answer_question",
    decide_citation,
    {"cite_regulation": "cite_regulation", "skip_citation": "skip_citation"},
)
builder.add_edge("cite_regulation", END)
builder.add_edge("skip_citation",   END)

graph = builder.compile()
print("Graph compiled.")

# ASCII representation
print(graph.get_graph().draw_ascii())

In [ ]:
# Run the agent
initial_state = {
    "messages": [HumanMessage(content="What coverage ratio must a Dutch pension fund maintain?")],
    "needs_citation": False,
    "citations": [],
}

final_state = graph.invoke(initial_state)

print("\n=== Final messages ===")
for msg in final_state["messages"]:
    role = msg.__class__.__name__
    print(f"\n[{role}]\n{msg.content}")

print("\nCitations:", final_state.get("citations", []))

## 6. Streaming — stream intermediate steps to the user

LangGraph supports streaming at multiple granularities:
- `stream_mode="updates"` — yields each node's state update as it happens
- `stream_mode="values"` — yields the full state after each node
- `stream_mode="messages"` — yields individual LLM tokens (requires streaming-capable LLM)


In [ ]:
print("=== Streaming node updates ===")
for chunk in graph.stream(initial_state, stream_mode="updates"):
    for node_name, update in chunk.items():
        msg_count = len(update.get("messages", []))
        print(f"  Node '{node_name}' returned {msg_count} new message(s)")
        if "needs_citation" in update:
            print(f"    needs_citation = {update['needs_citation']}")

## 7. Human-in-the-loop — interrupt for human approval

LangGraph can **pause** execution before a specified node and wait for human input.
This is done by:
1. Compiling with `interrupt_before=["node_name"]`
2. The graph will pause and return when it reaches that node
3. The host app collects human input and calls `graph.invoke()` again with the same thread config


In [ ]:
memory = MemorySaver()

# Compile with interrupt_before cite_regulation
graph_with_interrupt = builder.compile(
    checkpointer=memory,
    interrupt_before=["cite_regulation"],
)

config = {"configurable": {"thread_id": "demo-thread-1"}}

print("--- First invocation (will pause before cite_regulation) ---")
result = graph_with_interrupt.invoke(initial_state, config=config)

# Inspect current state
snapshot = graph_with_interrupt.get_state(config)
print(f"Next node to run: {snapshot.next}")
print(f"needs_citation:   {snapshot.values.get('needs_citation')}")

In [ ]:
# Simulate human approving the citation step
print("\n--- Human approves: resuming graph ---")

# Pass None to resume from the checkpoint without modifying state
resumed_result = graph_with_interrupt.invoke(None, config=config)

print("\n=== Final messages after resume ===")
for msg in resumed_result["messages"]:
    role = msg.__class__.__name__
    print(f"[{role}] {msg.content[:120]}")

## 8. Memory and persistence — using MemorySaver

`MemorySaver` stores all state snapshots in-memory, keyed by `thread_id`.
For production, swap it for `SqliteSaver` or `PostgresSaver`.


In [ ]:
# Multi-turn conversation with persistent memory
persistent_memory = MemorySaver()
persistent_graph = builder.compile(checkpointer=persistent_memory)

thread_config = {"configurable": {"thread_id": "pension-chat-42"}}

queries = [
    "What is IORP III?",
    "Which article covers the ORSA requirement?",
    "And what are the deadlines?",  # context-dependent — tests memory
]

for turn, query in enumerate(queries, 1):
    print(f"\n--- Turn {turn}: {query} ---")
    state = {"messages": [HumanMessage(content=query)], "needs_citation": False, "citations": []}
    result = persistent_graph.invoke(state, config=thread_config)
    last_ai = next(
        (m for m in reversed(result["messages"]) if isinstance(m, AIMessage)), None
    )
    if last_ai:
        print(last_ai.content[:300])

# Show conversation history
print("\n--- Full conversation history ---")
snapshot = persistent_graph.get_state(thread_config)
print(f"Total messages stored: {len(snapshot.values['messages'])}")

## 9. Key takeaways

| Concept | Summary |
|---------|--------|
| `StateGraph` | The core primitive — nodes + edges form the agent graph |
| State reducers | `add_messages` appends; default overwrites |
| Conditional edges | Route to different nodes based on a decision function |
| `interrupt_before` | Pause execution for human review before a node runs |
| `MemorySaver` | Persists state across calls using `thread_id` |
| Streaming | `stream_mode="updates"` yields node-level diffs in real time |

## Exercises

1. **Add a third tool** — `get_asset_allocation(fund: str)` that returns mock allocation data, and add a node that calls it when the user asks about "portfolio" or "allocation".
2. **Retry loop** — Add a cycle: if the LLM answer is shorter than 50 words, loop back and regenerate with a `be_more_detailed=True` state flag.
3. **Replace MemorySaver** — Swap `MemorySaver` for `SqliteSaver` and persist conversations across Python sessions.
4. **Stream tokens** — Enable token-level streaming using `astream_events` and print each token as it arrives.
5. **Full agent** — Import `pension_agent.py` from `../agent/` and run it against two different query types (regulation + fund metrics). Trace the routing decision.